# MP-Declare Constraint Mining

Mining MP-Declare constraints with data conditions from the BPIC17 event log using RuM's MINERful + MpEnhancer.

In [1]:
import sys
import os
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

from src.interpretability.perturbation_methods import csv_to_xes, discover_mpdeclare

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:163: DeprecationWarning: module 'sre_parse' is deprecated
  import sre_parse
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:164: DeprecationWarning: module 'sre_constants' is deprecated
  import sre_constants


In [2]:
# ===== TEST MODE =====
# Set to True to use a small subset of data for quick testing.
# This avoids Java OOM on the full BPIC17 log and speeds up VAE training.
TEST_MODE = True
CONFIG.use_improved = False  # set True for the improved variant
TEST_N_CASES = 100      # Number of cases for XES constraint mining
TEST_N_SEQUENCES = 50   # Number of test sequences to load

In [3]:
import pandas as pd
import pm4py

csv_path = _current / 'data' / 'BPI_Challenge_2017.csv'

if TEST_MODE:
    xes_path = _current / 'data' / 'BPI_Challenge_2017_test_subset.xes'
else:
    xes_path = _current / 'data' / 'BPI_Challenge_2017.xes'

if not xes_path.exists():
    df = pd.read_csv(csv_path)
    if TEST_MODE:
        subset_cases = df["case:concept:name"].unique()[:TEST_N_CASES]
        df = df[df["case:concept:name"].isin(subset_cases)]
        print(f"TEST MODE: Subsetting to {len(subset_cases)} cases ({len(df)} events)")
    # Fill NaN in ALL columns to avoid Java NumberFormatException when parsing XES
    df = df.dropna(axis=1, how='all')
    for col in df.select_dtypes(include='number').columns:
        df[col] = df[col].fillna(0)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].fillna('')
    df = pm4py.format_dataframe(df, case_id="case:concept:name", activity_key="concept:name", timestamp_key="time:timestamp")
    pm4py.write_xes(pm4py.convert_to_event_log(df), str(xes_path))
    print(f"Converted CSV to XES: {xes_path}")
else:
    print(f"XES file already exists: {xes_path}")

XES file already exists: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/data/BPI_Challenge_2017_test_subset.xes


In [4]:
# Start JVM with extra heap before discover_mpdeclare (avoids OOM on large logs)
import jpype
if not jpype.isJVMStarted():
    from src.interpretability.perturbation_methods.revised_plus.rum_mpdeclare import RUM_JAR
    jpype.startJVM(f"-Djava.class.path={RUM_JAR}", "-Djava.awt.headless=true", "-Xmx4g", convertStrings=True)
    __import__('jpype.imports')

constraints = discover_mpdeclare(xes_path, min_support=0.95, data_conditions='ACTIVATIONS')
print(f"Mined {len(constraints)} MP-Declare constraints")

log4j:WARN No appenders could be found for logger (minerful.miner.core.MinerFulKBCore).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


||||||||||||||||||||||||||||||||||||||||


[discover_mpdeclare] Skipping MpEnhancer: min_support (0.95) >= data_condition_threshold (0.9), no data conditions would be kept.
Mined 103 MP-Declare constraints


In [5]:
print("=" * 100)
print("MP-DECLARE CONSTRAINTS WITH DATA CONDITIONS")
print("=" * 100)
print()

with_data = [c for c in constraints if c.data_condition]
print(f"Found {len(with_data)} constraints with data conditions:\n")

for i, c in enumerate(with_data, 1):
    print(f"{i}. {c.template}[{c.activation}, {c.target}]")
    print(f"   Support: {c.support:.1%}")
    print(f"   Data condition: {c.data_condition}")
    print()

MP-DECLARE CONSTRAINTS WITH DATA CONDITIONS

Found 0 constraints with data conditions:



In [6]:
print("=" * 100)
print("ALL MINED CONSTRAINTS")
print("=" * 100)
print()

for i, c in enumerate(constraints, 1):
    print(f"{i:3}. {c}")

ALL MINED CONSTRAINTS

  1. Init[Activity: "A_Create Application" (supp=1.0)] (support=100.0%)
  2. Precedence[Activity: "A_Accepted" (supp=1.0), Activity: "A_Incomplete" (supp=1.0)] (support=100.0%)
  3. Precedence[Activity: "A_Accepted" (supp=1.0), Activity: "A_Validating" (supp=1.0)] (support=100.0%)
  4. Precedence[Activity: "A_Accepted" (supp=1.0), Activity: "O_Cancelled" (supp=1.0)] (support=100.0%)
  5. Succession[Activity: "A_Accepted" (supp=1.0), Activity: "O_Create Offer" (supp=1.0)] (support=100.0%)
  6. Succession[Activity: "A_Accepted" (supp=1.0), Activity: "O_Created" (supp=1.0)] (support=100.0%)
  7. Precedence[Activity: "A_Accepted" (supp=1.0), Activity: "O_Refused" (supp=1.0)] (support=100.0%)
  8. Precedence[Activity: "A_Accepted" (supp=1.0), Activity: "O_Returned" (supp=1.0)] (support=100.0%)
  9. Succession[Activity: "A_Accepted" (supp=1.0), Activity: "O_Sent (mail and online)" (supp=1.0)] (support=100.0%)
 10. Precedence[Activity: "A_Accepted" (supp=1.0), Activity:

# REVISED+ Counterfactual Explanations

Generate counterfactual prefixes using the REVISED+ orchestrator:
- VAE trained on **random-length prefixes** (learns the prefix manifold)
- **Two-tier plausibility**: prefix-safe constraints for search penalty, all constraints for validity gate
- Latent space elite-sampling search

**Note**: BPIC17 is significantly larger than Helpdesk/DomesticDeclarations (~6300 test cases, 9 categorical + 9 numerical features). VAE training will take longer on first run.

In [ ]:
import torch

import sys
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# --- Load dataset ---
data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
full_dataset = torch.load(data_path, weights_only=False)

if TEST_MODE:
    # Build a proper list of (cat_tuple, num_tuple, case_id) items
    # (slicing EventLogDataset directly returns a single batched tuple, not a list)
    dataset = [full_dataset[i] for i in range(min(TEST_N_SEQUENCES, len(full_dataset)))]
    print(f"TEST MODE: Using {len(dataset)} sequences (subset of {len(full_dataset)})")
else:
    dataset = full_dataset

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f"Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}")

# --- Load trained prediction model ---
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

from src.interpretability.config.bpic17_config import CONFIG
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [8]:
from src.interpretability.utils.tensor_decoder import TensorDecoder

# TensorDecoder needs the full dataset object (with .all_categories, .encoder_decoder)
decoder = TensorDecoder(full_dataset)

# Build activity_names list: index -> name (for the REVISED+ orchestrator)
# BPIC17 uses 'concept:name' as the activity feature (not 'Activity')
ACTIVITY_FEATURE = 'concept:name'

activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]

# Find EOS index
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

print(f"Activity vocabulary ({len(activity_names)}):")
for i, name in enumerate(activity_names):
    marker = ' <-- EOS' if i == eos_idx else ''
    print(f"  {i}: {name}{marker}")

Activity vocabulary (28):
  0: <pad>
  1: A_Accepted
  2: A_Cancelled
  3: A_Complete
  4: A_Concept
  5: A_Create Application
  6: A_Denied
  7: A_Incomplete
  8: A_Pending
  9: A_Submitted
  10: A_Validating
  11: EOS <-- EOS
  12: O_Accepted
  13: O_Cancelled
  14: O_Create Offer
  15: O_Created
  16: O_Refused
  17: O_Returned
  18: O_Sent (mail and online)
  19: O_Sent (online only)
  20: W_Assess potential fraud
  21: W_Call after offers
  22: W_Call incomplete files
  23: W_Complete application
  24: W_Handle leads
  25: W_Personal Loan collection
  26: W_Shortened completion 
  27: W_Validate application


In [9]:
from src.interpretability.perturbation_methods import RevisedPlus, RevisedPlusConfig, create_revised_plus_for_model

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

config = RevisedPlusConfig(
    vae_epochs=10 if TEST_MODE else 100,
    vae_kl_weight=0.1,
    declare_min_support=0.9,
    n_candidates_per_round=200,
    n_search_rounds=5,
    top_k=5,
    min_plausibility=0.0,
    device=device,
    activity_feature=ACTIVITY_FEATURE,
)

vae_path = str(_current / 'encoded_data' / ('bpic17_vae_test.pkl' if TEST_MODE else 'bpic17_vae.pkl'))

rp = create_revised_plus_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    vae_path=vae_path,
)
print(f"\nREVISED+ ready:")
print(f"  VAE parameters: {sum(p.numel() for p in rp.vae.parameters()):,}")
print(f"  VAE path: {vae_path}")
print(f"  All constraints: {len(rp.all_constraints)}")
print(f"  Prefix-safe constraints: {len(rp.prefix_safe_constraints)}")

Extracting activity sequences...


Mined 778 constraints (520 prefix-safe)
Loading VAE from /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/encoded_data/bpic17_vae_test.pkl

REVISED+ ready:
  VAE parameters: 202,061
  VAE path: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/encoded_data/bpic17_vae_test.pkl
  All constraints: 778
  Prefix-safe constraints: 520


In [10]:
# === Debug: inspect VAE samples and reconstructions ===
import torch

def _act_seq(tensor):
    """Convert activity tensor to list of names, stripping padding."""
    return [activity_names[x.item()] for x in tensor if x.item() != 0]

vae = rp.vae
vae.eval()
dev = next(vae.parameters()).device

# --- 1) Sample from the prior (z ~ N(0,1)) ---
print("=" * 80)
print("VAE SAMPLES FROM PRIOR (z ~ N(0,1))")
print("=" * 80)
with torch.no_grad():
    cat_sampled, num_sampled = vae.sample(10, device=dev)
for i in range(10):
    seq = _act_seq(cat_sampled[0][i])
    print(f"  Sample {i+1:2d} (len={len(seq):2d}): {' -> '.join(seq)}")

# --- 2) Reconstruct real prefixes (encode to mu, decode) ---
print(f"\n{'=' * 80}")
print("VAE RECONSTRUCTIONS (encode -> mu -> decode)")
print("=" * 80)
n_show = min(10, len(dataset))
for i in range(n_show):
    cat_tuple, num_tuple, case_id = dataset[i]
    cat_in = [c.unsqueeze(0).to(dev) for c in cat_tuple]
    num_in = torch.stack(num_tuple, dim=-1).unsqueeze(0).to(dev)

    orig_seq = _act_seq(cat_tuple[0])

    with torch.no_grad():
        cat_recon, _ = vae.reconstruct(cat_in, num_in)
    recon_seq = _act_seq(cat_recon[0][0])

    match = orig_seq == recon_seq
    print(f"  Case {i+1:2d} ({'MATCH' if match else 'DIFF'}):")
    print(f"    Original (len={len(orig_seq):2d}): {' -> '.join(orig_seq)}")
    if not match:
        print(f"    Recon    (len={len(recon_seq):2d}): {' -> '.join(recon_seq)}")

# --- 3) Encode a real prefix, perturb z, decode (mimics search) ---
print(f"\n{'=' * 80}")
print("LATENT PERTURBATIONS (encode -> z + noise -> decode)")
print("=" * 80)
cat_tuple, num_tuple, _ = dataset[0]
cat_in = [c.unsqueeze(0).to(dev) for c in cat_tuple]
num_in = torch.stack(num_tuple, dim=-1).unsqueeze(0).to(dev)
orig_seq = _act_seq(cat_tuple[0])
print(f"  Original (len={len(orig_seq):2d}): {' -> '.join(orig_seq)}")

with torch.no_grad():
    mu, logvar = vae.encode(cat_in, num_in)
    for noise_scale in [0.1, 0.5, 1.0, 2.0]:
        z = mu + noise_scale * torch.randn_like(mu)
        cat_logits, _ = vae.decode(z)
        cat_decoded = [l.argmax(dim=-1) for l in cat_logits]
        perturbed_seq = _act_seq(cat_decoded[0][0])
        print(f"  noise={noise_scale:.1f} (len={len(perturbed_seq):2d}): {' -> '.join(perturbed_seq)}")

VAE SAMPLES FROM PRIOR (z ~ N(0,1))


  Sample  1 (len=96): O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -> O_Sent (mail and online) -

  Sample  9 (len=96): A_Denied -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completion  -> W_Shortened completio

  Case  1 (DIFF):
    Original (len= 5): A_Create Application -> A_Submitted -> W_Handle leads -> W_Handle leads -> W_Complete application
    Recon    (len=96): A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -

  Case  7 (DIFF):
    Original (len=11): A_Create Application -> A_Submitted -> W_Handle leads -> W_Handle leads -> W_Complete application -> A_Concept -> W_Complete application -> W_Complete application -> A_Accepted -> O_Create Offer -> O_Created
    Recon    (len=96): A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied

  noise=0.1 (len=96): A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Denied -> A_Deni

In [11]:
print("PREFIX-SAFE constraints (used in search penalty):")
print("-" * 60)
for c in sorted(rp.prefix_safe_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

print(f"\nNOT prefix-safe constraints (used in validity gate):")
print("-" * 60)
not_safe = rp.all_constraints - rp.prefix_safe_constraints
for c in sorted(not_safe, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

PREFIX-SAFE constraints (used in search penalty):
------------------------------------------------------------
  absence(A_Denied, n=1)
  absence(O_Refused, n=1)
  alternate_precedence(A_Accepted, A_Cancelled)
  alternate_precedence(A_Accepted, A_Complete)
  alternate_precedence(A_Accepted, A_Denied)
  alternate_precedence(A_Accepted, A_Validating)
  alternate_precedence(A_Accepted, EOS)
  alternate_precedence(A_Accepted, O_Cancelled)
  alternate_precedence(A_Accepted, O_Create Offer)
  alternate_precedence(A_Accepted, O_Created)
  alternate_precedence(A_Accepted, O_Refused)
  alternate_precedence(A_Accepted, O_Returned)
  alternate_precedence(A_Accepted, O_Sent (mail and online))
  alternate_precedence(A_Cancelled, A_Denied)
  alternate_precedence(A_Cancelled, O_Cancelled)
  alternate_precedence(A_Cancelled, O_Refused)
  alternate_precedence(A_Complete, A_Cancelled)
  alternate_precedence(A_Complete, A_Denied)
  alternate_precedence(A_Complete, A_Validating)
  alternate_precedence(A_C

In [12]:
import numpy as np

# Find a good candidate: in-progress prefix with uncertain prediction
best_idx, best_prob = None, 1.0
search_limit = min(50 if TEST_MODE else 200, len(dataset))
for i in range(search_limit):
    cat_t, num_t, _ = dataset[i]
    act = cat_t[0]  # concept:name is the first categorical feature
    # Skip completed traces (contain EOS)
    if (act == eos_idx).any():
        continue
    cat_in = [c.unsqueeze(0) for c in cat_t]
    num_in = [n.unsqueeze(0) for n in num_t]
    with torch.no_grad():
        preds = model((cat_in, num_in))[0]
        logits = preds[0][f"{ACTIVITY_FEATURE}_mean"][0]
        p = torch.softmax(logits, dim=-1)
        top_p = p.max().item()
    if 0.4 < top_p < best_prob:
        best_idx, best_prob = i, top_p

test_idx = best_idx if best_idx is not None else 42
print(f"Selected test_idx={test_idx} (top_p={best_prob:.3f})")

cat_tuple, num_tuple, case_id = dataset[test_idx]

print(f"\nCase: {case_id}")
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

explanation = rp.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

Selected test_idx=48 (top_p=0.418)

Case: Application_1000806256


,concept:name,Action,org:resource,EventOrigin,lifecycle:transition,case:LoanGoal,case:ApplicationType,Accepted,Selected,case_elapsed_time,event_elapsed_time,day_in_week,seconds_in_day,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,MonthlyCost,CreditScore,Case ID
0,A_Create Application,Created,User_4,Application,complete,Existing loan takeover,New credit,<pad>,<pad>,0.000000e+00,50897.105469,2.0,32626.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
1,W_Complete application,Created,User_4,Workflow,schedule,Existing loan takeover,New credit,<pad>,<pad>,0.000000e+00,0.011719,2.0,32626.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
2,W_Complete application,Obtained,User_4,Workflow,start,Existing loan takeover,New credit,<pad>,<pad>,0.000000e+00,0.003906,2.0,32626.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
3,A_Concept,statechange,User_4,Application,complete,Existing loan takeover,New credit,<pad>,<pad>,0.000000e+00,0.003906,2.0,32626.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
4,W_Complete application,Released,User_4,Workflow,suspend,Existing loan takeover,New credit,<pad>,<pad>,2.953125e+02,295.289062,2.0,32922.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
5,W_Complete application,Obtained,User_4,Workflow,resume,Existing loan takeover,New credit,<pad>,<pad>,5.042625e+03,4747.343750,2.0,37669.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
6,A_Accepted,statechange,User_4,Application,complete,Existing loan takeover,New credit,<pad>,<pad>,5.800250e+03,757.609375,2.0,38427.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
7,O_Create Offer,Created,User_4,Offer,complete,Existing loan takeover,New credit,True,True,5.902375e+03,102.140625,2.0,38529.0,0.0,0.000000,120.000000,624.169983,0.000000,Application_1000806256
8,O_Created,statechange,User_4,Offer,complete,Existing loan takeover,New credit,<pad>,<pad>,5.903875e+03,1.484375,2.0,38530.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256
9,O_Sent (mail and online),statechange,User_4,Offer,complete,Existing loan takeover,New credit,<pad>,<pad>,5.944625e+03,40.765625,2.0,38571.0,0.0,8383.772461,82.958527,280.393005,319.191345,Application_1000806256


Original: W_Validate application (p=0.418), prefix_len=29


REVISED+ Counterfactual Explanation
Original: W_Validate application (idx=27, p=0.418)
Prefix length: 29 events
Target: any different class
Constraints: 520 prefix-safe, 778 total
Search: 1000 evaluated, 0 valid, 0.2s

No counterfactuals found.


In [13]:
if explanation.counterfactuals:
    best = explanation.get_best()

    print("=" * 80)
    print("ORIGINAL PREFIX")
    print(f"Prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print(f"Prefix length: {explanation.prefix_len} events")
    print("=" * 80)
    display(df_orig)

    print(f"\n{'=' * 80}")
    print("BEST COUNTERFACTUAL PREFIX")
    print(f"Prediction: {best.counterfactual_prediction_name} (p={best.counterfactual_probability:.3f})")
    cf_prefix_len = len(best.activity_sequence)
    print(f"Prefix length: {cf_prefix_len} events (delta={cf_prefix_len - explanation.prefix_len:+d})")
    print(f"Proximity: {best.proximity:.3f}  Sparsity: {best.sparsity}")
    print(f"Feasibility: {best.feasibility:.3f}")
    print(f"Plausibility (definite): {best.plausibility_definite:.2f}")
    print(f"Plausibility (optimistic): {best.plausibility_optimistic:.2f}")
    print(f"Combined score: {best.combined_score:.4f}")
    print("=" * 80)

    cf_cat = tuple(best.cat_sequence)
    cf_num = tuple(best.num_sequence[:, i] for i in range(best.num_sequence.shape[1])) if best.num_sequence is not None else num_tuple
    df_cf = decoder.decode_sequence(cf_cat, cf_num, skip_padding=False)
    display(df_cf)
else:
    print("No counterfactuals found. Try increasing n_search_rounds or noise_scale.")

No counterfactuals found. Try increasing n_search_rounds or noise_scale.


In [14]:
import pandas as pd

if explanation.counterfactuals:
    rows = []
    for i, cf in enumerate(explanation.counterfactuals):
        rows.append({
            'rank': i + 1,
            'prediction': cf.counterfactual_prediction_name,
            'probability': f"{cf.counterfactual_probability:.3f}",
            'prefix_len': len(cf.activity_sequence),
            'activities': ' -> '.join(activity_names[a] for a in cf.activity_sequence),
            'proximity': f"{cf.proximity:.2f}",
            'sparsity': cf.sparsity,
            'feasibility': f"{cf.feasibility:.3f}",
            'plaus_def': f"{cf.plausibility_definite:.2f}",
            'plaus_opt': f"{cf.plausibility_optimistic:.2f}",
            'score': f"{cf.combined_score:.4f}",
        })

    df_cfs = pd.DataFrame(rows)
    print(f"Original: {' -> '.join(activity_names[a] for a in explanation.original_activity_sequence)}")
    print(f"Original prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print()
    display(df_cfs)